# Camp QMIND mini demo: train a sparse autoencoder, then tune a feature up and down

Individual, self-contained demo for [MVP.md](MVP.md) on **Qwen3-0.6B-Base**. Instead of downloading a
pretrained SAE, we train our own, and make its dictionary **100x wider than the model's hidden size**
(1024 -> 102,400 features). Steps: load the model, collect hidden states, train the SAE (a plain training
loop on two linear layers), find a feature for a concept, see what it fires on, and clamp it up and down while
the model writes. How the clamp works is explained in [steering.md](steering.md).

Needs a GPU with roughly 8 GB free. CPU works for a smoke test (shrink `EXPANSION` and `N_TOKENS`) but the
full settings are slow there. Nothing here has been run at full scale. The probe sentences are drafts nobody
has read, so read them before trusting any result. Per [AGENTS.md](AGENTS.md), do not quote numbers from this
notebook unless you ran it.

**Expect a rough SAE.** A 100x-wide dictionary trained on about a million tokens will have many dead
features and unclean ones. Real SAEs train on billions of tokens. That is fine for seeing the mechanics, but
it is exactly why the audit in MVP.md exists.

In [5]:
import random
import torch
import torch.nn as nn
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = "Qwen/Qwen3-0.6B-Base"
LAYER = 14          # a middle layer (of 28); a guess, try a few
EXPANSION = 50     # SAE width = EXPANSION * hidden size. The brief asks for at least 100
K = 50              # TopK sparsity: only the K strongest features stay active per token
BATCH = 1024        # SAE training batch (tokens). Raise it if you have GPU memory to spare
EPOCHS = 4
LR = 2e-4
CONCEPT = "kingston" 

## 1. Load the model and data

In [6]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cpu


In [7]:
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.bfloat16).to(device).eval()
D_MODEL = model.config.hidden_size
D_SAE = EXPANSION * D_MODEL
print(f"hidden size {D_MODEL}, SAE width {D_SAE:,} ({EXPANSION}x)")

docs = (d["text"] for d in load_dataset("HuggingFaceFW/fineweb", "sample-10BT", split="train", streaming=True))

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

hidden size 1024, SAE width 51,200 (50x)


Resolving data files:   0%|          | 0/27468 [00:00<?, ?it/s]

## 2. Collect hidden states

We stream FineWeb text and save the hidden state after block `LAYER` for every token. That is the same place
the steering hook will act later. A hook stops the forward pass right after that block, so we skip the
upper half of the model. We drop each text's first token: it acts as an attention sink with a huge,
unrepresentative norm. Finally we divide by one constant so that typical vectors have norm about
sqrt(hidden size), which keeps the learning rate sensible.

In [ ]:
N_TOKENS = 10_000  # tokens of FineWeb to train on. Far too few for a good SAE, enough for a demo
class Stop(Exception):
    pass

def grab_hidden(batch_ids):
    """Hidden state after block LAYER, shape (batch, seq, d_model). Stops the forward pass there."""
    saved = {}
    def hook(_m, _i, out):
        saved["h"] = out[0] if isinstance(out, tuple) else out
        raise Stop
    handle = model.model.layers[LAYER].register_forward_hook(hook)
    try:
        with torch.no_grad():
            model(**batch_ids)
    except Stop:
        pass
    finally:
        handle.remove()
    return saved["h"]

chunks, n = [], 0
while n < N_TOKENS:
    texts = [next(docs) for _ in range(16)]
    ids = tok(texts, return_tensors="pt", padding=True, truncation=True, max_length=256).to(device)
    h = grab_hidden(ids)[:, 1:]            # drop the first token of every text
    keep = ids["attention_mask"][:, 1:].bool()
    chunks.append(h[keep].to("cpu", torch.float16))
    n += chunks[-1].shape[0]
    if len(chunks) % 50 == 0:
        print(f"{n:,} tokens")
data = torch.cat(chunks)[:N_TOKENS]
SCALE = data.float().norm(dim=-1).mean().item() / D_MODEL ** 0.5
print(f"{data.shape[0]:,} vectors of size {data.shape[1]}, SCALE = {SCALE:.2f}")

10,000 vectors of size 1024, SCALE = 1.71


: 

## 3. Train the SAE

The SAE is two linear layers. The encoder maps a hidden vector (1024 numbers) up to 102,400 numbers and keeps
only the K largest (the top-k). The decoder maps those back down to 1024. Training minimises the squared
error between the input and the rebuilt vector. Each decoder column (one feature's direction) is kept at
unit length, so a feature's activation is its strength and not a size trade-off with its direction.
Everything else is a standard training loop.

In [ ]:
import gc
del model   # the LM is not needed while training the SAE; free its ~1.2 GB (reloaded before the helpers below)
gc.collect()
if device == "cuda":
    torch.cuda.empty_cache()

class SAE(nn.Module):
    def __init__(self, d_model, d_sae, k):
        super().__init__()
        self.k = k
        self.enc = nn.Linear(d_model, d_sae)
        self.dec = nn.Linear(d_sae, d_model)
        with torch.no_grad():
            self.dec.weight.copy_(nn.functional.normalize(torch.randn(d_model, d_sae), dim=0))  # unit columns
            self.enc.weight.copy_(self.dec.weight.T)   # start the encoder as the decoder's transpose
            self.enc.bias.zero_()
            self.dec.bias.zero_()

    def encode(self, x):
        pre = self.enc(x - self.dec.bias)
        vals, idx = pre.topk(self.k, dim=-1)
        return torch.zeros_like(pre).scatter_(-1, idx, vals.relu())

    def decode(self, acts):
        return self.dec(acts)

    def forward(self, x):
        acts = self.encode(x)
        return self.decode(acts), acts

sae = SAE(D_MODEL, D_SAE, K).to(device)
with torch.no_grad():
    sae.dec.bias.copy_(data[:10000].float().mean(0).to(device) / SCALE)   # start the bias at the data mean
opt = torch.optim.Adam(sae.parameters(), lr=LR)

fired = torch.zeros(D_SAE, dtype=torch.bool, device=device)   # features that fired since the last print
step = 0
for epoch in range(EPOCHS):
    perm = torch.randperm(data.shape[0])
    for i in range(0, data.shape[0] - BATCH + 1, BATCH):
        x = (data[perm[i:i + BATCH]].float() / SCALE).to(device)
        recon, acts = sae(x)
        loss = (recon - x).pow(2).sum(-1).mean()
        opt.zero_grad()
        loss.backward()
        opt.step()
        with torch.no_grad():   # keep every decoder column at unit length
            sae.dec.weight.copy_(nn.functional.normalize(sae.dec.weight, dim=0))
            fired |= (acts > 0).any(0)
        step += 1
        unexplained = ((recon - x).pow(2).sum() / (x - x.mean(0)).pow(2).sum()).item()
        print(f"epoch {epoch} step {step}: loss {loss.item():.2f}, variance unexplained {unexplained:.3f}, "
                f"features that fired in the last 50 steps {fired.float().mean().item():.1%}")
        fired.zero_()
sae.eval()
del opt   # drop Adam's state (two extra copies of the weights) now that training is done
gc.collect()

W1003 14:39:47.071000 1190 torch/autograd/profiler.py:858] Unexpected error importing torch.profiler._cupti_monitor
W1003 14:39:47.071000 1190 torch/autograd/profiler.py:858] Traceback (most recent call last):
W1003 14:39:47.071000 1190 torch/autograd/profiler.py:858]   File "/home/k6/Github/test-time-reasoning/.venv/lib/python3.12/site-packages/torch/autograd/profiler.py", line 852, in _maybe_cupti_monitor
W1003 14:39:47.071000 1190 torch/autograd/profiler.py:858]     return all_function_events
W1003 14:39:47.071000 1190 torch/autograd/profiler.py:858]         ^^^^^^^^^^^^^^^^^^^^^^^
W1003 14:39:47.071000 1190 torch/autograd/profiler.py:858] ImportError: cannot import name '_cupti_monitor' from 'torch.profiler' (/home/k6/Github/test-time-reasoning/.venv/lib/python3.12/site-packages/torch/profiler/__init__.py)


epoch 0 step 1: loss 457.44, variance unexplained 0.750, features that fired in the last 50 steps 55.4%
epoch 0 step 2: loss 450.65, variance unexplained 0.744, features that fired in the last 50 steps 55.7%
epoch 0 step 3: loss 454.12, variance unexplained 0.740, features that fired in the last 50 steps 56.0%
epoch 0 step 4: loss 442.85, variance unexplained 0.736, features that fired in the last 50 steps 56.1%
epoch 0 step 5: loss 450.75, variance unexplained 0.731, features that fired in the last 50 steps 56.3%
epoch 0 step 6: loss 436.91, variance unexplained 0.728, features that fired in the last 50 steps 56.5%
epoch 0 step 7: loss 440.15, variance unexplained 0.725, features that fired in the last 50 steps 56.8%
epoch 0 step 8: loss 435.67, variance unexplained 0.722, features that fired in the last 50 steps 56.9%
epoch 0 step 9: loss 440.46, variance unexplained 0.719, features that fired in the last 50 steps 57.5%
epoch 1 step 10: loss 419.20, variance unexplained 0.678, featur

Helpers. `encode` turns a hidden vector into sparse feature activations, `decode` turns them back,
`residual` grabs the hidden state after a layer, and `clamp` is the steering hook. Activations live in the
scaled units the SAE was trained in, and `clamp` converts back to real units when it edits the hidden state.

In [ ]:
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.bfloat16).to(device).eval()   # reload the LM freed before training

def encode(x):
    with torch.no_grad():
        return sae.encode(x.float() / SCALE)

def decode(acts):
    with torch.no_grad():
        return sae.decode(acts) * SCALE

def residual(text, layer=LAYER):
    """Hidden state after block `layer` (first token dropped, as in training) and the tokens."""
    ids = tok(text, return_tensors="pt").to(device)
    with torch.no_grad():
        hidden = model(**ids, output_hidden_states=True).hidden_states
    return hidden[layer + 1][0, 1:], tok.convert_ids_to_tokens(ids["input_ids"][0])[1:]  # hidden[0] is embeddings

def generate(prompt, n=1000):
    ids = tok(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(**ids, max_new_tokens=n, do_sample=False)
    return tok.decode(out[0, ids["input_ids"].shape[1]:], skip_special_tokens=True)

def clamp(feature, value, layer=LAYER):
    """Set one feature to `value` (SAE units) at every position. Returns a handle; call .remove() to undo."""
    def hook(_module, _inputs, output):
        h = output[0] if isinstance(output, tuple) else output
        act = encode(h)[..., feature]
        h2 = h + ((value - act).unsqueeze(-1) * sae.dec.weight[:, feature].detach() * SCALE).to(h.dtype)
        return (h2,) + tuple(output[1:]) if isinstance(output, tuple) else h2
    return model.model.layers[layer].register_forward_hook(hook)

## How good is the SAE?

Reconstruction error on a sentence the SAE never saw (relative to the vector's size), how many features are
still dead, and how many are active per token. **If the error is near 1 the SAE learned little: train longer
or on more tokens before reading anything into the features below.** A large dead fraction is expected here.

In [ ]:
text = "The Golden Gate Bridge opened in 1937 and spans the strait at San Francisco."
x, _ = residual(text)
err = ((x.float() - decode(encode(x))).norm() / x.float().norm()).item()
print(f"relative reconstruction error on a held-out sentence: {err:.3f}")
with torch.no_grad():
    probe = (data[:20000].float() / SCALE).to(device)
    alive = torch.cat([(sae.encode(probe[i:i + 1000]) > 0).any(0, keepdim=True) for i in range(0, 20000, 1000)]).any(0)
print(f"features that fire on at least one of 20,000 training tokens: {alive.float().mean().item():.1%} "
      f"of {D_SAE:,}; active per token: {K}")

## 4. Find a feature for a concept

Each concept has **target** sentences (should fire), **near-miss** sentences (close in topic, should not
fire) and shared **neutral** sentences. We score every feature by its mean peak activation on target minus
neutral, then show the near-miss value next to it. A good candidate is high on target and low on both others.

In [ ]:
NEUTRAL = [
    "The recipe calls for two cups of flour and a pinch of salt.",
    "The train to the airport leaves every twenty minutes.",
    "She repaired the bicycle chain with a small wrench.",
    "The stock market closed slightly higher on Tuesday.",
    "He practiced the piano for an hour before dinner.",
    "The new phone has a larger battery and a faster camera.",
    "They painted the fence white before the neighbours arrived.",
    "The committee will meet again on the first Monday of March.",
    "The library closes at nine on weekdays.",
    "He added a teaspoon of cinnamon to the oatmeal.",
    "The printer ran out of ink halfway through the report.",
    "A light rain fell on the garden all afternoon.",
    "The bus was ten minutes late this morning.",
    "She folded the laundry and put it in the drawer.",
    "The museum offers free admission on the first Sunday of the month.",
    "He set an alarm for six o'clock.",
    "The software update takes about five minutes to install.",
    "Farmers expect a good harvest after the wet spring.",
    "The dog waited by the door for its walk.",
    "They compared prices at three different stores.",
]
PROBES = {
    "queens": {
        "target": [
            "Queen's University in Kingston, Ontario was founded in 1841.",
            "She spent four years at Queen's studying engineering and living near campus.",
            "Frosh Week at Queen's starts with students in tricolour jackets.",
            "The Queen's Gaels won the football game at Richardson Stadium.",
            "Grant Hall is the clock tower building at the heart of the Queen's campus.",
            "He applied to Queen's Commerce and hoped to study at Goodes Hall.",
            "Homecoming at Queen's brings thousands of alumni back to Kingston.",
            "Stauffer Library stays packed during exams at Queen's.",
            "Queen's University is located in Kingston, Ontario, on the shore of Lake Ontario.",
            "Queen's received its royal charter from Queen Victoria in 1841.",
            "Queen's students cheer for the Gaels in blue, gold and red.",
            "The Queen's Faculty of Law is housed in Macdonald Hall.",
            "My sister is in her second year at Queen's and loves living in Kingston.",
            "She rented a house near campus while studying nursing at Queen's.",
            "Students at Queen's study in Douglas Library and Stauffer Library during exams.",
            "The Queen's Gaels basketball team played a home game at the ARC.",
            "Biology students at Queen's do fieldwork at the Queen's University Biological Station.",
            "Applicants to Queen's must submit their grades before the January deadline.",
            "A chemistry degree from Queen's led her to a career in research.",
            "The Queen's Principal welcomed the new class at the opening convocation.",
        ],
        "near_miss": [
            "Queen Elizabeth II was crowned in 1953 at Westminster Abbey.",
            "The band Queen played Wembley Stadium in 1986.",
            "The queen bee lays up to two thousand eggs a day.",
            "In chess, the queen can move any number of squares in a straight line.",
            "The University of Toronto is the largest university in Ontario.",
            "Waterloo is known for its co-op engineering programs.",
            "Kingston has a famous limestone waterfront and a historic penitentiary.",
            "McGill University is in Montreal and is known for its research.",
            "Queens is a borough of New York City and is home to LaGuardia and JFK airports.",
            "Queen Victoria reigned over the United Kingdom for more than sixty years.",
            "The Queen of Hearts shouts for heads to be cut off in Alice in Wonderland.",
            "Queens of the Stone Age released the album Songs for the Deaf in 2002.",
            "Western University is in London, Ontario.",
            "McMaster University is in Hamilton and has a well-known medical school.",
            "Carleton University is in Ottawa.",
            "The Royal Military College of Canada is also in Kingston.",
            "Lake Ontario is the smallest of the Great Lakes by surface area.",
            "The Thousand Islands are a short drive east of Kingston.",
            "In a deck of cards, the queen ranks below the king.",
            "Queen Anne's lace is a wildflower common in Ontario meadows.",
        ],
    },
    # "bridge": {
    #     "target": [
    #         "The Golden Gate Bridge opened in 1937 and spans the strait at San Francisco Bay.",
    #         "We drove across the Golden Gate Bridge as fog rolled in from the Pacific.",
    #         "The Golden Gate Bridge is painted a colour called International Orange.",
    #         "Thousands of tourists walk the Golden Gate Bridge every day.",
    #         "From Marin County you can see the Golden Gate Bridge towers rising out of the mist.",
    #         "Joseph Strauss was the chief engineer of the Golden Gate Bridge.",
    #         "The Golden Gate Bridge was the longest suspension bridge in the world when it opened.",
    #         "Cyclists cross the Golden Gate Bridge to reach Sausalito.",
    #         "The Golden Gate Bridge connects San Francisco to Marin County.",
    #         "Construction of the Golden Gate Bridge began in 1933.",
    #         "The two towers of the Golden Gate Bridge are joined by thick steel cables.",
    #         "Fog often hides the tops of the Golden Gate Bridge towers in the morning.",
    #         "Visitors photograph the Golden Gate Bridge from Battery Spencer.",
    #         "The Golden Gate Bridge has a pedestrian walkway on its east side.",
    #         "Drivers heading south across the Golden Gate Bridge pay a toll.",
    #         "The Golden Gate Bridge crosses the mouth of San Francisco Bay.",
    #         "Painters work year-round to keep the Golden Gate Bridge from rusting.",
    #         "Her wedding photos were taken at Crissy Field with the Golden Gate Bridge behind them.",
    #         "The art deco towers of the Golden Gate Bridge have vertical ribs.",
    #         "Ships sail under the Golden Gate Bridge on their way into the bay.",
    #     ],
    #     "near_miss": [
    #         "The Brooklyn Bridge connects Manhattan and Brooklyn across the East River.",
    #         "The Bay Bridge links San Francisco to Oakland.",
    #         "Alcatraz Island sits in the middle of San Francisco Bay.",
    #         "Fog is common along the California coast in summer.",
    #         "The Tower Bridge in London opens to let ships pass.",
    #         "Cable cars climb the steep hills of San Francisco.",
    #         "The Sydney Harbour Bridge is nicknamed the Coathanger.",
    #         "Lombard Street in San Francisco is famous for its sharp curves.",
    #         "The Verrazzano-Narrows Bridge links Staten Island and Brooklyn.",
    #         "The Akashi Kaikyo Bridge in Japan is one of the longest suspension bridges in the world.",
    #         "Muir Woods is a redwood forest north of San Francisco.",
    #         "Pier 39 is a popular tourist spot known for its sea lions.",
    #         "The Ambassador Bridge connects Detroit and Windsor.",
    #         "Suspension bridges hang the roadway from large cables strung between towers.",
    #         "The Mackinac Bridge links the two peninsulas of Michigan.",
    #         "The Presidio is a former military base in San Francisco.",
    #         "Engineers inspect the steel cables of old bridges for corrosion.",
    #         "The Rialto Bridge is a famous stone bridge over the Grand Canal in Venice.",
    #         "Sausalito is a small waterfront town in Marin County.",
    #         "Chinatown in San Francisco is one of the oldest in North America.",
    #     ],
    # },
}

sets = {**PROBES[CONCEPT], "neutral": NEUTRAL}
peak = {name: torch.stack([encode(residual(s)[0]).max(0).values for s in sents]) for name, sents in sets.items()}
mean = {name: p.mean(0) for name, p in peak.items()}
score = mean["target"] - mean["neutral"]

print(f"{'feature':>8} {'score':>8} {'target':>8} {'near':>8} {'neutral':>8} {'max seen':>9}")
best = score.topk(10).indices.tolist()
for f in best:
    top = max(p[:, f].max().item() for p in peak.values())
    print(f"{f:>8} {score[f]:>8.2f} {mean['target'][f]:>8.2f} {mean['near_miss'][f]:>8.2f} "
          f"{mean['neutral'][f]:>8.2f} {top:>9.2f}")

Pick a feature from the table. By default we take the top-scoring one, but look at the near-miss column
first: a feature that fires just as hard on the near-miss sentences is probably the wrong one.

In [ ]:
FEATURE = best[0]  # change this to another index from the table if the top one looks wrong
MAX_ACT = max(p[:, FEATURE].max().item() for p in peak.values())
print(f"feature {FEATURE}, max activation seen {MAX_ACT:.2f}")

## 5. What does it actually fire on?

For each sentence, the token where the feature peaks. Do this before believing any label for the feature.

In [ ]:
for name in ("target", "near_miss"):
    print(f"--- {name} ---")
    for s in sets[name]:
        x, tokens = residual(s)
        acts = encode(x)[:, FEATURE]
        i = acts.argmax().item()
        print(f"{acts[i].item():6.1f}  on {tokens[i]!r:<14} {s}")

## 6. Steer it

We generate from the same prompt with no steering, then with the feature clamped at several fractions of its
max (0 removes it), then with a **random feature** clamped just as hard as a control. If the random feature
does the same thing, the effect is just "we pushed the hidden state somewhere strange". Greedy decoding, one
prompt: read every output, it is a demo and not evidence.

In [ ]:
PROMPT = "Q: What should I study in university?\nA:"
MULTIPLES = [0, 0.5, 1, 2, 5]

print("--- no steering ---")
print(generate(PROMPT), "\n")
for m in MULTIPLES:
    handle = clamp(FEATURE, m * MAX_ACT)
    out = generate(PROMPT)
    handle.remove()
    print(f"--- feature {FEATURE} at {m}x max ---\n{out}\n")

control = random.Random(0).randrange(D_SAE)
handle = clamp(control, max(MULTIPLES) * MAX_ACT)
out = generate(PROMPT)
handle.remove()
print(f"--- CONTROL: random feature {control} at {max(MULTIPLES)}x ---\n{out}")

### To "tune it down"

Use a prompt where the feature should matter, such as `"Q: Tell me about Queen's University in Kingston.\nA:"`,
and compare no steering with `m = 0`. If the answer gets vaguer or drifts to a generic university, the
feature is doing causal work. Also try the other concept (`CONCEPT = "bridge"`) and a prompt-only baseline
("Talk about the Golden Gate Bridge in every answer.") to see whether plain prompting beats steering, as
AxBench reports.

### Caveats

- This SAE is small-data and short-trained. Many of its 102,400 features are dead, and a concept as narrow as
  Queen's University may have no clean feature at all. Train on more tokens before concluding either way.
- The layer is an unverified guess, and the SAE was trained on one layer only.
- Probe sets are tiny and unreviewed. A feature that tops 8 sentences can still be a mislabel, which is the
  point of the audit in MVP.md.
- Setting a feature to 0 removes one direction only. A weak "down" effect does not mean the feature is
  unimportant (see steering.md).